In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.nn as jnn
import jax.tree_util as jtu
import jax.linalg as jla
from jax import jit, vmap
from jaxtyping import Array, Float, Int, PyTree  # https://github.com/google/jaxtyping

In [ ]:
class NN_Build(eqx.Module):
    layers: list

    def __init__(self, 
                 input_shape: tuple,
                 output_dim: int,
                 layer_dims: list,
                 activation: Callable = jnn.relu,
                 key: jax.random.PRNGKey = None,
                 ) :
        # PyTree structure for the layers of the neural network. The layers will be stored in a list, and each layer will be added according to the dimensions specified in layer_dims.
        # NamedTuple structure for the layers of the neural network.
        self.layers = []

        # layer_dimss is a list of tuples, where each tuple represents the size of the layer. 
        # For a linear layer, the tuple will have two elements (input_dim, output_dim). 
        # For a convolutional layer, the tuple will have three elements (input_channels, output_channels, kernel_size).
        self.layer_dims = jtu.tree_map(lambda x: tuple(x) if not isinstance(x, tuple) else x, layer_dims)
        if len(self.layer_dims) == 0:
            raise ValueError("At least one layer dimension must be specified in layer_dims.")
        self.activation = activation
        self.key = jax.random.PRNGKey(0) if key is None else key
        self.input_shape = input_shape
        self.output_dim = output_dim

        # Check if the layer dimensions are compatible with the input shape and output dimension. This function will raise an error if the dimensions do not match.
        self.check_layer_dims(self.input_shape, self.output_dim, self.layer_dims)

        # Model building loop according to the layer dimensions provided. The first layer is added separately to handle the input shape, and then the rest of the layers are added in a loop.
        if len(self.layer_dims) == 1:
            self._add_layer(self.layer_dims[0], None, self.layers, final_layer=True)
        else:
            self._add_layer(self.layer_dims[0], None, self.layers, final_layer=False)
            size_prev = self.layer_dims[0]
            for size_new in self.layer_dims[1:-1]:
                self._add_layer(size_new, size_prev, self.layers, final_layer=False)
                size_prev = size_new
            self._add_layer(self.layer_dims[-1], size_prev, self.layers, final_layer=True)
            size_prev = self.layer_dims[-1]


    def check_layer_dims(self,
                         input_shape: tuple,
                         output_dim: int,
                         layer_dims: list):
        # The function checks if the final data shape after passing through all layers matches the specified output dimension.
        # data_shape is the actual shape of the data as it passes through the layers of the neural network. 
        data_shape = input_shape
        for layer_dim in layer_dims:
            if len(layer_dim) not in [2, 3]:
                raise ValueError(f"Each layer dimension must be a tuple of length 2 (for linear layers) or 3 (for convolutional layers). Got {layer_dim}.")
            elif len(layer_dim) == 2:
                flattened_shape = jnp.prod(data_shape)
                if flattened_shape != layer_dim[0]:
                    raise ValueError(f"Expected input shape {data_shape} to be compatible with linear layer input dimension {layer_dim[0]}.")
                data_shape = (layer_dim[1],)
            elif len(layer_dim) == 3:
                if len(data_shape) == 3:
                    if data_shape[2] != layer_dim[0]:
                        raise ValueError(f"Expected input shape {data_shape} to be compatible with convolutional layer input channels {layer_dim[0]}.")
                    new_h = data_shape[0] - (layer_dim[2] - 1)
                    new_w = data_shape[1] - (layer_dim[2] - 1)
                    if new_h <= 0 or new_w <= 0:
                        raise ValueError(f"Kernel size {layer_dim[2]} is too large for input shape {data_shape}.")
                    data_shape = (new_h, new_w, layer_dim[1])
                elif len(data_shape) != 3:
                    raise ValueError(f"Cannot add a convolutional layer after a linear layer, got layer dimensions {layer_dim} after input shape {data_shape}.")
        if data_shape != (output_dim,):
            raise ValueError(f"Final layer shape {data_shape} does not match output dimension {output_dim}.")

    def _add_layer(self, 
                   layer_dim: tuple, 
                   prev_layer_dim: tuple, 
                   layers: list,
                   final_layer: bool = False):
        # Special case for the first layer with no previous layer dimensions.
        if prev_layer_dim is None:
            if len(layer_dim) == 2:
                if len(self.input_shape) == 1:
                    if self.input_shape[0] != layer_dim[0]:
                        raise ValueError(
                            f"Expected input shape {self.input_shape} to match linear "
                            f"input dimension {layer_dim[0]}."
                        )
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=self.key))
                elif len(self.input_shape) == 3:
                    if prod(self.input_shape) != layer_dim[0]:
                        raise ValueError(
                            f"Expected flattened input size {prod(self.input_shape)} to "
                            f"match linear input dimension {layer_dim[0]}."
                        )
                    layers.append(eqx.nn.Flatten())
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=self.key))
            elif len(layer_dim) == 3: 
                if self.input_shape[2] != layer_dim[0]:
                    raise ValueError(f"Expected input shape {self.input_shape} to be compatible with convolutional layer input channels {layer_dim[0]}.")   
                elif len(self.input_shape) == 3:     
                    layers.append(eqx.nn.Conv2d(layer_dim[0], layer_dim[1], kernel_size=layer_dim[2], key=self.key))
                elif len(self.input_shape) != 3:
                    raise ValueError(f"Cannot add a convolutional layer after a linear layer, got layer dimensions {layer_dim} after input shape {self.input_shape}.")
        # Adding rest of the layers according to the dimensions specified in layer_dim and prev_layer_dim. 
        # The function checks the dimensions and adds the appropriate layer (linear or convolutional) followed by the activation function.
        # No padding, stride of 1, and no dilation are assumed for convolutional layers.
        else:
            if len(layer_dim) == 2: 
                if len(prev_layer_dim) == 2:
                    if prev_layer_dim[1] != layer_dim[0]:
                        raise ValueError(f"Expected previous layer output dimension {prev_layer_dim[1]} to match linear input dimension {layer_dim[0]}.")
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=self.key))
                elif len(prev_layer_dim) == 3:
                    layers.append(eqx.nn.Flatten())
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=self.key))
            elif len(layer_dim) == 3:
                if len(prev_layer_dim) == 2:
                    raise ValueError(f"Cannot add a convolutional layer after a linear layer, got layer dimensions {layer_dim} after {prev_layer_dim}.")
                elif len(prev_layer_dim) == 3:
                    if prev_layer_dim[1] != layer_dim[0]:
                        raise ValueError(f"Expected previous layer output channels {prev_layer_dim[1]} to match linear input dimension {layer_dim[0]}.")
                    layers.append(eqx.nn.Conv2d(layer_dim[0], layer_dim[1], kernel_size=layer_dim[2], key=self.key))
        # If this is not the final layer, we add the activation function.
        if not final_layer:
            layers.append(self.activation)

    def __call__(self, x):
        for l, layer in enumerate(self.layers):
            x = layer(x)
        return x
    


    